# 04 Outbreak Investigation Workflow — Reference Solutions

Complete solutions to the Songbai Nursing Home Legionnaires' disease cluster SitRep exercises.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (avoid CJK labels rendering as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Summary metrics

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Convert dates
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("Songbai Nursing Home Legionnaires' Disease Cluster — SitRep")
print("=" * 50)
print(f"Total residents: {total}")
print(f"Infected: {infected} (attack rate {infected/total:.1%})")
print(f"  Confirmed: {confirmed}   Probable: {probable}")
print(f"Hospitalized: {hospitalized} (hospitalization rate {hospitalized/infected:.1%})")
print(f"ICU: {icu} (ICU rate {icu/hospitalized:.1%})")
print(f"Deaths: {deaths} (CFR {deaths/infected:.1%})")

## Question 2: The person/time/place trio

In [ ]:
# --- Person ---
cases = df[df["infected"] == 1]

print("=== Demographic characteristics (infected) ===")
print(f"Median age: {cases['age'].median():.0f} years"
      f" (range {cases['age'].min()}-{cases['age'].max()})")
print(f"Proportion male: {(cases['sex'] == 'M').mean():.1%}")

In [ ]:
import matplotlib.dates as mdates

# --- Time ---
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# Add the pre-outbreak baseline period (including zero-case days)
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Epidemic Curve, by Onset Date, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Use the original daily series (without the baseline period) to find the peak
daily_cases = cases.groupby("symptom_onset_date").size().rename("cases")
print(f"Outbreak period: {daily_cases.index.min().date()} – {daily_cases.index.max().date()}")
print(f"Peak day: {daily_cases.idxmax().date()} ({daily_cases.max()} cases)")

In [ ]:
# --- Place ---
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
)
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== Outbreak summary by wing ===")
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))

## Question 3: Stratified summary by age group

In [ ]:
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

age_stats = (
    df.groupby("age_group", observed=True)
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
)
age_stats["AR%"] = (age_stats["infected"] / age_stats["residents"] * 100).round(1)
age_stats["CFR%"] = (age_stats["deaths"] / age_stats["infected"] * 100).round(1)

print("=== Age group stratified summary ===")
print(age_stats.to_string())

print(f"\nHighest attack rate: {age_stats['AR%'].idxmax()} ({age_stats['AR%'].max()}%)")
print(f"Highest CFR: {age_stats['CFR%'].idxmax()} ({age_stats['CFR%'].max()}%)")
print("→ The age group with the highest attack rate isn't necessarily the one with the highest CFR —")
print("  the attack rate reflects 'risk of infection', while CFR reflects 'prognosis after infection'; the two are driven by different factors.")

## Question 4 (challenge): the generate_sitrep function

In [ ]:
def generate_sitrep(csv_path):
    """Produce a SitRep summary dictionary from a CSV."""
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    infected_n = int(df["infected"].sum())
    deaths_n = int((df["outcome"] == "dead").sum())

    # peak day
    cases = df[df["infected"] == 1]
    daily = cases.groupby("symptom_onset_date").size()
    peak_date = str(daily.idxmax().date()) if len(daily) > 0 else None

    # wing with the highest attack rate
    ws = (
        df.groupby(["floor", "wing"])
        .agg(residents=("case_id", "size"), infected=("infected", "sum"))
        .reset_index()
    )
    ws["ar"] = ws["infected"] / ws["residents"]
    worst = ws.loc[ws["ar"].idxmax()]
    worst_wing = f"{worst['floor']}{worst['wing']}"

    return {
        "total_residents": total,
        "infected": infected_n,
        "attack_rate": round(infected_n / total * 100, 1),
        "deaths": deaths_n,
        "cfr": round(deaths_n / infected_n * 100, 1) if infected_n else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
        "peak_date": peak_date,
        "worst_wing": worst_wing,
    }

result = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print("=== SitRep structured output ===")
for k, v in result.items():
    print(f"  {k}: {v}")

## Question 5: Produce a Word report

In [ ]:
from io import BytesIO
from datetime import datetime
from docx import Document
from docx.shared import Inches

pathlib.Path("output").mkdir(exist_ok=True)

# Save the epidemic curve into memory
epicurve_buf = BytesIO()
fig.savefig(epicurve_buf, format="png", dpi=150, bbox_inches="tight")
epicurve_buf.seek(0)

report_time = datetime.now().strftime("%Y-%m-%d %H:%M")

doc = Document()
doc.add_heading("Songbai Nursing Home Legionnaires' SitRep", level=1)
doc.add_paragraph(f"Report time: {report_time}")

# Summary metrics table
doc.add_heading("Summary metrics", level=2)
table = doc.add_table(rows=5, cols=2, style="Light Grid Accent 1")
for i, (label, value) in enumerate([
    ("Total residents", str(total)),
    ("Infected", f"{infected} (attack rate {infected/total:.1%})"),
    ("Confirmed / Probable", f"{confirmed} / {probable}"),
    ("Hospitalized / ICU", f"{hospitalized} / {icu}"),
    ("Deaths", f"{deaths} (CFR {deaths/infected:.1%})"),
]):
    table.rows[i].cells[0].text = label
    table.rows[i].cells[1].text = value

# Embed the epidemic curve
doc.add_heading("Epidemic curve", level=2)
epicurve_buf.seek(0)
doc.add_picture(epicurve_buf, width=Inches(6))

doc.save("output/my_sitrep.docx")
print("Word report saved: output/my_sitrep.docx")

### Interpretation

- **Attack rate ~43%**: very high, indicating a severe outbreak with a widespread exposure source
- **CFR ~16%**: Legionnaires' disease has an elevated CFR in nursing-home populations, consistent with the literature
- **Wing 3B has the highest attack rate**: prioritize investigating that wing's water supply and shower equipment
- **Age group differences**: the highest attack rate and the highest CFR may not fall in the same age group, showing that "risk of infection" and "prognosis" are driven by different factors